# 05 — Basket Recommendations with Spark FP-Growth

This notebook uses Spark FP-Growth to mine frequent product combinations and association rules from Instacart baskets. The goal is to create interpretable cross-sell recommendations that complement the personalized reorder prediction model from Notebook 04.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.fpm import FPGrowth

In [0]:
base_path = "/Volumes/workspace/default/instacart"  # change if needed
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

dbutils.fs.mkdirs(gold_path)

In [0]:
prior = spark.read.parquet(f"{silver_path}/prior_order_details")
products = spark.read.parquet(f"{silver_path}/product_details")

## 1. Build transaction baskets

In [0]:
transactions = (
    prior
    .groupBy("order_id")
    .agg(
        F.collect_set("product_id").alias("items"),
        F.countDistinct("product_id").alias("basket_size")
    )
    .filter(F.col("basket_size") >= 2)
)

display(transactions.limit(10))

In [0]:
display(
    transactions
    .select("basket_size")
    .summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max")
)

In [0]:
transactions.write.mode("overwrite").parquet(f"{gold_path}/basket_transactions")

In [0]:
transactions_filtered = (
    transactions
    .filter((F.col("basket_size") >= 2) & (F.col("basket_size") <= 50))
)

print("Original transactions:", transactions.count())
print("Filtered transactions:", transactions_filtered.count())

### Why FP-Growth?

A naive market basket analysis approach would require one-hot encoding thousands of products across millions of orders, creating a very large sparse matrix. Spark FP-Growth avoids this by representing each order as an array of product IDs and mining frequent itemsets directly in a distributed format.

## 3. Train FP-Growth

### Practical sampling decision

Running FP-Growth on the full transaction table exceeded Databricks Free Edition/serverless compute limits. Since this notebook is intended to generate interpretable basket-level cross-sell rules, FP-Growth is trained on a representative sample of historical baskets. The full-scale personalized prediction task is handled by the supervised reorder model.

In [0]:
# Use a practical sample for Databricks Free Edition
transactions_model = (
    transactions_filtered
    .sample(withReplacement=False, fraction=0.25, seed=42)
)

print("Sampled transactions:", transactions_model.count())

fp = FPGrowth(
    itemsCol="items",
    minSupport=0.003,
    minConfidence=0.25
)

fp_model = fp.fit(transactions_model)

## 4. Frequent itemsets

In [0]:
freq_itemsets = fp_model.freqItemsets
display(freq_itemsets.orderBy(F.desc("freq")).limit(50))

In [0]:
print("Frequent itemsets:", freq_itemsets.count())

In [0]:
freq_itemsets.write.mode("overwrite").parquet(f"{gold_path}/frequent_itemsets")

## 5. Association rules

In [0]:
association_rules = fp_model.associationRules

display(
    association_rules
    .orderBy(F.desc("lift"), F.desc("confidence"))
    .limit(50)
)

In [0]:
print("Association rules:", association_rules.count())

In [0]:
association_rules.write.mode("overwrite").parquet(f"{gold_path}/association_rules_raw")

### Association rule metrics

- **Support**: how often the product combination appears in all baskets.
- **Confidence**: probability of buying the consequent product given the antecedent product(s).
- **Lift**: how much more often the consequent appears with the antecedent than expected by chance.

For business recommendations, lift and confidence are both important. High lift identifies meaningful relationships, while confidence ensures the recommendation is common enough to be useful.

## 6. Filter useful rules

Raw association rules can be noisy or difficult to interpret. The final rules are filtered to keep simple, business-friendly recommendations: at most two products in the basket trigger and exactly one recommended product.

In [0]:
rules_filtered = (
    association_rules
    .withColumn("antecedent_size", F.size("antecedent"))
    .withColumn("consequent_size", F.size("consequent"))
    .filter(F.col("antecedent_size") <= 2)
    .filter(F.col("consequent_size") == 1)
    .filter(F.col("confidence") >= 0.20)
    .filter(F.col("lift") >= 1.2)
    .orderBy(F.desc("lift"), F.desc("confidence"))
)

display(rules_filtered.limit(50))

In [0]:
print("Filtered useful rules:", rules_filtered.count())

In [0]:
rules_filtered.write.mode("overwrite").parquet(f"{gold_path}/association_rules_filtered")

## 7. Convert product IDs to product names

### 7.1 Product lookup

In [0]:
product_lookup = products.select(
    "product_id",
    "product_name",
    "aisle",
    "department"
).dropDuplicates(["product_id"])

### 7.2 Name antecedents

In [0]:
antecedent_names = (
    rules_filtered
    .withColumn("antecedent_product_id", F.explode("antecedent"))
    .join(
        product_lookup.withColumnRenamed("product_id", "antecedent_product_id")
                      .withColumnRenamed("product_name", "antecedent_product_name")
                      .withColumnRenamed("aisle", "antecedent_aisle")
                      .withColumnRenamed("department", "antecedent_department"),
        on="antecedent_product_id",
        how="left"
    )
    .groupBy("antecedent", "consequent", "confidence", "lift", "support", "antecedent_size", "consequent_size")
    .agg(
        F.collect_set("antecedent_product_name").alias("antecedent_names"),
        F.collect_set("antecedent_aisle").alias("antecedent_aisles"),
        F.collect_set("antecedent_department").alias("antecedent_departments")
    )
)

### 7.3 Name consequents

In [0]:
rules_named = (
    antecedent_names
    .withColumn("consequent_product_id", F.explode("consequent"))
    .join(
        product_lookup.withColumnRenamed("product_id", "consequent_product_id")
                      .withColumnRenamed("product_name", "consequent_product_name")
                      .withColumnRenamed("aisle", "consequent_aisle")
                      .withColumnRenamed("department", "consequent_department"),
        on="consequent_product_id",
        how="left"
    )
    .select(
        "antecedent",
        "antecedent_names",
        "antecedent_aisles",
        "antecedent_departments",
        "consequent",
        "consequent_product_id",
        "consequent_product_name",
        "consequent_aisle",
        "consequent_department",
        "confidence",
        "lift",
        "support"
    )
    .orderBy(F.desc("lift"), F.desc("confidence"))
)

display(rules_named.limit(50))

In [0]:
rules_named.write.mode("overwrite").parquet(f"{gold_path}/association_rules_named")

## 8. Create business-friendly cross-sell recommendations

In [0]:
cross_sell_recommendations = (
    rules_named
    .select(
        F.concat_ws(", ", F.col("antecedent_names")).alias("basket_contains"),
        F.col("consequent_product_name").alias("recommend_product"),
        F.col("consequent_aisle").alias("recommend_aisle"),
        F.col("consequent_department").alias("recommend_department"),
        "confidence",
        "lift",
        "support"
    )
    .orderBy(F.desc("lift"), F.desc("confidence"))
)

display(cross_sell_recommendations.limit(30))

In [0]:
cross_sell_recommendations.write.mode("overwrite").parquet(
    f"{gold_path}/cross_sell_recommendations"
)

The cross-sell table converts association rules into a business-readable format: "if the basket contains these products, recommend this product." These outputs can support checkout add-ons, bundle suggestions, and department-specific merchandising strategies.

## 9. Department-specific rules

In [0]:
produce_rules = (
    rules_named
    .filter(F.array_contains(F.col("antecedent_departments"), "produce"))
    .orderBy(F.desc("lift"), F.desc("confidence"))
)

display(produce_rules.limit(20))

In [0]:
dairy_rules = (
    rules_named
    .filter(F.array_contains(F.col("antecedent_departments"), "dairy eggs"))
    .orderBy(F.desc("lift"), F.desc("confidence"))
)

display(dairy_rules.limit(20))

In [0]:
produce_rules.write.mode("overwrite").parquet(f"{gold_path}/produce_association_rules")
dairy_rules.write.mode("overwrite").parquet(f"{gold_path}/dairy_association_rules")

## 10. Generate recommendations for a sample basket

In [0]:
sample_basket = (
    transactions_model
    .filter(F.col("basket_size").between(5, 15))
    .limit(1)
)

display(sample_basket)

In [0]:
sample_recs = fp_model.transform(sample_basket)
display(sample_recs)

In [0]:
sample_recs_exploded = (
    sample_recs
    .select("order_id", F.explode("prediction").alias("product_id"))
    .join(product_lookup, on="product_id", how="left")
)

display(sample_recs_exploded)

In [0]:
sample_recs_exploded.write.mode("overwrite").parquet(
    f"{gold_path}/sample_fpgrowth_recommendations"
)

## FP-Growth Basket Recommendation Summary

This notebook uses Spark FP-Growth to mine frequent itemsets and association rules from historical Instacart baskets. Unlike dense one-hot market basket encoding, FP-Growth represents each order as an array of product IDs, making the approach scalable for millions of order-product records.

The final association rules are filtered for interpretability using:

- Antecedent size <= 2
- Consequent size = 1
- Confidence >= 0.20
- Lift >= 1.2

These rules support cross-sell recommendations such as: when a basket contains product A, recommend product B. Lift is especially useful because it identifies product relationships that occur more often than expected by chance.

This FP-Growth component complements the reorder prediction model:

- Reorder model: personalized next-basket prediction based on user history
- FP-Growth: interpretable basket-level cross-sell recommendations based on product co-occurrence